# Fitness Survey OCR Pipeline - Hybrid (OpenCV bubble detection + Gemini)
Local-only Jupyter notebook. No Colab / Google Drive dependency.

Bubble/checkbox fields calibrated in `bubble_templates_blue.json` are read
via OpenCV (fast, free, deterministic). Everything else - and any calibrated
field whose detection confidence is too low - falls back to Gemini.

Run order: Config -> Field definitions -> ROI helpers -> Bubble detection ->
Gemini helpers -> Hybrid extraction -> (optional) Calibration tool -> Main run -> Export.


In [ ]:
!pip install -q pymupdf pillow openpyxl pandas google-generativeai opencv-python-headless numpy ipympl ipywidgets


In [ ]:
import os
import json
import time
import glob

import fitz
import cv2
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font
from PIL import Image
import google.generativeai as genai

PDF_INPUT_DIR = "./input_pdfs"
OUT_DIR = "./ocr_output"
BUBBLE_TEMPLATE_PATH = "./bubble_templates_blue.json"
DPI = 300

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
MODEL_NAME = "gemini-3.5-flash-lite"

SLEEP_BETWEEN_CALLS = 5.0      # throttle to stay under free-tier RPM limits
OPENCV_CONF_THRESHOLD = 0.75   # below this, fall back to Gemini for that field
DARK_THRESH_OFFSET = 40        # adaptive darkness threshold offset
GEMINI_SELF_CONSISTENT = False # True = call Gemini twice per group to cross-check (2x cost)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PDF_INPUT_DIR, exist_ok=True)

if not GEMINI_API_KEY:
    print("GEMINI_API_KEY not set (see README).")
else:
    genai.configure(api_key=GEMINI_API_KEY)
print(f"Model: {MODEL_NAME}")


In [ ]:
# All 87 output fields, in fixed order.
COLUMN_NAMES = [
    "学校名", "性別", "No.", "握力_右", "握力_左", "上体起こし", "長座体前屈", "反復横とび",
    "持久走_分", "持久走_秒", "20mシャトルラン", "50m走", "立ち幅とび", "ハンドボール投げ", "身長", "体重",
    "質1", "質2-❶", "質2-❷", "質2-❸", "質2-❹", "質2-❺", "質3",
    "質4-❶", "質4-❷", "質4-❸", "質4-❹", "質4-❺", "質4-❻その他", "質5",
    "質6-❶", "質6-❷", "質6-❸", "質7-①", "質7-②", "質7-③", "質7-④", "質7-⑤",
    "質8_部活動_月", "質8_部活動_火", "質8_部活動_水", "質8_部活動_木", "質8_部活動_金", "質8_部活動_土", "質8_部活動_日",
    "質8_地域_月", "質8_地域_火", "質8_地域_水", "質8_地域_木", "質8_地域_金", "質8_地域_土", "質8_地域_日",
    "質8_それ以外_月", "質8_それ以外_火", "質8_それ以外_水", "質8_それ以外_木", "質8_それ以外_金", "質8_それ以外_土", "質8_それ以外_日",
    "質8-2-①", "質8-2-②", "質8-2-③", "質8-2-④", "質8-2-⑤", "質8-2-⑤その他",
    "質9", "質10", "質11", "質12",
    "質12-2-①", "質12-2-②", "質12-2-③", "質12-2-④", "質12-2-⑤", "質12-2-⑥", "質12-2-⑦", "質12-2-⑧", "質12-2-⑨", "質12-2-⑩", "質12-2-⑩その他",
    "質13", "質14", "質15", "質16", "質17", "質18", "質19"
]
assert len(COLUMN_NAMES) == 87

# ROI groups matching physical form layout.
# Page = one spread: LEFT half (質8~19), RIGHT half (header + 実技 + 質1~7).
GROUP_HEADER_MEASURE = ["学校名","性別","No.","握力_右","握力_左","上体起こし","長座体前屈","反復横とび",
                         "持久走_分","持久走_秒","20mシャトルラン","50m走","立ち幅とび","ハンドボール投げ","身長","体重"]

GROUP_SURVEY_1_7 = ["質1","質2-❶","質2-❷","質2-❸","質2-❹","質2-❺","質3","質4-❶","質4-❷","質4-❸",
                     "質4-❹","質4-❺","質4-❻その他","質5","質6-❶","質6-❷","質6-❸",
                     "質7-①","質7-②","質7-③","質7-④","質7-⑤"]

GROUP_CLUB_TIME = ["質8_部活動_月","質8_部活動_火","質8_部活動_水","質8_部活動_木","質8_部活動_金","質8_部活動_土","質8_部活動_日",
                    "質8_地域_月","質8_地域_火","質8_地域_水","質8_地域_木","質8_地域_金","質8_地域_土","質8_地域_日",
                    "質8_それ以外_月","質8_それ以外_火","質8_それ以外_水","質8_それ以外_木","質8_それ以外_金","質8_それ以外_土","質8_それ以外_日",
                    "質8-2-①","質8-2-②","質8-2-③","質8-2-④","質8-2-⑤","質8-2-⑤その他"]

GROUP_DAILY_HABIT = ["質9","質10","質11"]

GROUP_HEALTH_CLASS = ["質12","質12-2-①","質12-2-②","質12-2-③","質12-2-④","質12-2-⑤","質12-2-⑥",
                       "質12-2-⑦","質12-2-⑧","質12-2-⑨","質12-2-⑩","質12-2-⑩その他",
                       "質13","質14","質15","質16","質17","質18","質19"]

ALL_GROUPS = [
    ("header_measure", GROUP_HEADER_MEASURE),
    ("survey_1_7", GROUP_SURVEY_1_7),
    ("club_time", GROUP_CLUB_TIME),
    ("daily_habit", GROUP_DAILY_HABIT),
    ("health_class", GROUP_HEALTH_CLASS),
]
assert sorted(sum([g for _, g in ALL_GROUPS], [])) == sorted(COLUMN_NAMES)

# ROI boundaries as fraction of each half-page height. Adjust if your scan layout differs.
ROI_FRACTIONS = {
    "header_measure": ("right", 0.00, 0.325),
    "survey_1_7":      ("right", 0.285, 1.00),
    "club_time":       ("left",  0.00, 0.29),
    "daily_habit":     ("left",  0.25, 0.435),
    "health_class":    ("left",  0.42, 1.00),
}

# FIELD_HINTS: real printed question text bound to each field key, so the model
# anchors on form content instead of guessing from surrounding rows/columns.
# Groups "質7", "質8-2", "質12-2" use a multi-column checkbox layout where the
# printed numbering goes left-to-right per row, then down (row-major) -
# NOT top-to-bottom per column. Mislabeling this order causes selected marks
# to land on the wrong field key.
FIELD_HINTS = {
    "質1": "運動やスポーツをすることは好きですか。選択肢:好き/やや好き/やや嫌い/嫌い",
    "質2-❶": "❶運動やスポーツをすること。選択肢:ある/ややある/あまりない/ない",
    "質2-❷": "❷運動やスポーツをみること",
    "質2-❸": "❸運動やスポーツを支えること(大会運営のボランティア等)",
    "質2-❹": "❹運動やスポーツを知ること(話を聞く、調べる等)",
    "質2-❺": "❺運動やスポーツを通じて様々な人と交流したり一体感を感じたりすること",
    "質3": "運動やスポーツをして楽しいと感じますか。選択肢:感じる/やや感じる/あまり感じない/感じない",
    "質4-❶": "❶体を動かしてすっきりした気分になったとき",
    "質4-❷": "❷様々な種目を体験したとき",
    "質4-❸": "❸できなかったことができるようになったとき",
    "質4-❹": "❹記録に挑戦したり、記録が伸びたり、思い通りに動けたとき",
    "質4-❺": "❺友達と交流したり、協力できたとき",
    "質4-❻その他": "❻その他(自由記述の手書きテキストをそのまま転記。空欄ならnull)",
    "質5": "卒業後も自主的に運動やスポーツをする時間を持ちたいと思いますか。選択肢:思う/やや思う/あまり思わない/思わない",
    "質6-❶": "❶家の人と一緒に運動すること。選択肢:ある/ややある/あまりない/ない",
    "質6-❷": "❷家の人とスポーツの話をすること",
    "質6-❸": "❸家の人とスポーツ観戦をすること(テレビ観戦を含む)",
    # 質7: single row, 5 marks. Note ③ is ONE combined label (スポーツ枠+その他スポーツクラブ), not two.
    "質7-①": "①学校の運動部(0/1)",
    "質7-②": "②学校の文化部(0/1)",
    "質7-③": "③地域クラブ活動(スポーツ)、その他のスポーツクラブ ― 1つの選択肢として扱う(0/1)",
    "質7-④": "④地域クラブ活動(文化)(0/1)",
    "質7-⑤": "⑤所属していない(0/1)",
    # 質8-2: 2-column x 2-row grid + その他, numbered row-major (left→right, then next row).
    "質8-2-①": "①運動する時間がないから(0/1、行1左)",
    "質8-2-②": "②運動する場所がないから(0/1、行1右)",
    "質8-2-③": "③一緒に運動する友達がいないから(0/1、行2左)",
    "質8-2-④": "④運動が好きではないから(0/1、行2右)",
    "質8-2-⑤": "⑤その他(0/1、最終行)。チェックされていれば右の自由記述欄に手書き理由があるはずなので、"
                "その内容は「質8-2-⑤その他」キーに転記すること",
    "質8-2-⑤その他": "質8-2-⑤「その他」の横に書かれた自由記述欄の手書きテキスト。"
                    "⑤が0(未チェック)、または欄が空欄の場合はnull",
    "質12": "保健体育の授業は楽しいですか。選択肢:楽しい/やや楽しい/あまり楽しくない/楽しくない",
    # 質12-2: 2-column x 5-row grid + その他, numbered row-major (left→right, then next row).
    "質12-2-①": "①運動のポイントを分かりやすく教えてもらえたら(0/1、行1左)",
    "質12-2-②": "②できなかったことができるようになったら(0/1、行1右)",
    "質12-2-③": "③自分に合った場やルールが用意されていたら(0/1、行2左)",
    "質12-2-④": "④タブレットなどのICTを活用できたら(0/1、行2右)",
    "質12-2-⑤": "⑤先生にほめてもらえたら(0/1、行3左)",
    "質12-2-⑥": "⑥友達に認めてもらえたら(0/1、行3右)",
    "質12-2-⑦": "⑦先生に個別に指導してもらえたら(0/1、行4左)",
    "質12-2-⑧": "⑧自分に合ったペースで行うことができたら(0/1、行4右)",
    "質12-2-⑨": "⑨できる・できないだけで比較されなかったら(0/1、行5左、右列なし)",
    "質12-2-⑩": "⑩その他(0/1、最終行)。チェックされていれば横の自由記述欄に手書き内容があるはずなので、"
                "その内容は「質12-2-⑩その他」キーに転記すること",
    "質12-2-⑩その他": "質12-2-⑩「その他」の横に書かれた自由記述欄の手書きテキスト。"
                     "⑩が0(未チェック)、または欄が空欄の場合はnull",
    "質13": "自分から「やってみたい」と思うときがありますか。選択肢:いつもある/だいたいある/あまりない/全くない",
    "質14": "目標(ねらい)を意識して学習することがありますか。同じ4択",
    "質15": "友達と助け合ったり、教え合ったりして学習することがありますか。同じ4択",
    "質16": "うまくいかないことがあったとき、考え話し合って取り組みましたか。選択肢:いつも取り組めていた/だいたい取り組めていた/あまり取り組めていなかった/取り組めていなかった",
    "質17": "タブレットなどのICTを活用して学習しましたか。同じ4択(いつもある〜全くない)",
    "質18": "運動・食事・休養・睡眠に気をつけた生活を送っていると思いますか。選択肢:思う/やや思う/あまり思わない/思わない",
    "質19": "保健を学習して、もっと運動しようと思いましたか。同じ4択(思う〜思わない)",
    "反復横とび": "④反復横とび。単位は「点(回)」。手書きの数字のみ(例: 29)",
    "ハンドボール投げ": "⑧ハンドボール投げ。単位は「m」。手書きの数字のみ(例: 4)",
    "身長": "(1)身長。単位は「cm」。小数第1位まで(例: 160.5)",
    "体重": "(2)体重。単位は「kg」。小数第1位まで(例: 56.9)",
    "握力_右": "①握力・右。単位 kg",
    "握力_左": "①握力・左。単位 kg",
    "上体起こし": "②上体起こし。単位 回",
    "長座体前屈": "③長座体前屈。単位 cm",
    "50m走": "⑥50m走。単位 秒",
    "立ち幅とび": "⑦立ち幅とび。単位 cm",
    "20mシャトルラン": "⑥20mシャトルラン。単位 回",
    "持久走_分": "⑤持久走の「分」。20mシャトルランとの選択種目のため、実施していなければ空欄(null)",
    "持久走_秒": "⑤持久走の「秒」。20mシャトルランとの選択種目のため、実施していなければ空欄(null)",
}

# "その他" checkbox fields paired with a free-text field capturing the handwritten note.
TEXT_COMPANION_GROUPS = {"survey_1_7", "club_time", "health_class"}

# Groups containing a multi-column (row-major) checkbox grid.
GRID_CHECKLIST_GROUPS = {"club_time", "health_class"}


In [ ]:
import fitz
import numpy as np
from PIL import Image

def render_page(pdf_path, page_num, dpi=DPI):
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    zoom = dpi / 72
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc.close()
    return img

def crop_roi(full_page_img, side, frac_top, frac_bottom):
    """side: 'left' or 'right' half of the page; frac_top/bottom: height ratio (0-1)."""
    w, h = full_page_img.size
    x0, x1 = (0, w // 2) if side == "left" else (w // 2, w)
    y0, y1 = int(h * frac_top), int(h * frac_bottom)
    return full_page_img.crop((x0, y0, x1, y1))

def get_all_rois(full_page_img):
    rois = {}
    for group_name, (side, top, bot) in ROI_FRACTIONS.items():
        rois[group_name] = crop_roi(full_page_img, side, top, bot)
    return rois


In [ ]:
def build_schema(fields):
    return {
        "type": "object",
        "properties": {f: {"type": "string", "nullable": True} for f in fields},
        "required": fields,
    }

def build_prompt(fields, group_name):
    hint_lines = [f'- "{f}": {FIELD_HINTS[f]}' for f in fields if f in FIELD_HINTS]
    hint_block = ""
    if hint_lines:
        hint_block = "各キーが対応する設問文（フォーム上の印刷文）:\n" + "\n".join(hint_lines) + "\n"

    anti_shift = ""
    if group_name in ("survey_1_7", "health_class"):
        anti_shift = (
            "【重要・厳守】各設問は横一列に4つの選択肢が並んでいます。"
            "必ず「この設問番号の行」に実際に塗られている／マークされている選択肢だけを読み取り、"
            "前後の設問のパターンから類推して埋めないでください。設問ごとに独立して判定してください。\n"
        )

    grid_order = ""
    if group_name in GRID_CHECKLIST_GROUPS:
        grid_order = (
            "【重要・厳守】チェックリスト形式の設問(質8-2、質12-2)は2列レイアウトです。"
            "番号は「左列から右列へ、その後次の行へ」の順（行優先）で振られています。"
            "列ごとに上から下へ読み進めないでください。各キーに紐づく設問文(上記)と"
            "画像上の実際の印刷テキストを照合し、正しい行・列のチェック有無だけを読み取ってください。\n"
        )

    anti_overmark = ""
    if group_name == "survey_1_7":
        anti_overmark = (
            "質7は複数選択可能なチェック欄ですが、実際にはっきりと黒い点／塗りつぶしがある項目のみ1、"
            "それ以外は全て0としてください。薄い印刷線や枠を選択マークと誤認しないでください。\n"
        )

    text_companion = ""
    if group_name in TEXT_COMPANION_GROUPS:
        text_companion = (
            "【重要】「その他」のチェック欄の横または下には、手書きの自由記述欄があります。"
            "対応するチェックが1(あり)の場合、その手書きテキストを対応する「...その他」キーに"
            "そのまま転記してください。小さい文字も見落とさず確認してください。"
            "チェックが0、または記述欄が空欄の場合はnullとしてください。\n"
        )

    return f"""あなたはOCRのプロフェッショナルです。添付画像から次のキーの値を正確に読み取り、JSONのみ返してください。
読み取れない・空欄は null。
{hint_block}{anti_shift}{grid_order}{anti_overmark}{text_companion}
キー一覧: {json.dumps(fields, ensure_ascii=False)}"""

def call_gemini_group(pil_image, fields, group_name, temperature=0.0):
    model = genai.GenerativeModel(MODEL_NAME)
    prompt = build_prompt(fields, group_name)
    resp = model.generate_content(
        [prompt, pil_image],
        generation_config={
            "temperature": temperature,
            "response_mime_type": "application/json",
            "response_schema": build_schema(fields),
        },
    )
    time.sleep(SLEEP_BETWEEN_CALLS)
    return json.loads(resp.text)

def call_group_self_consistent(pil_image, fields, group_name):
    """Call Gemini once, or twice with cross-check if GEMINI_SELF_CONSISTENT is True."""
    r1 = call_gemini_group(pil_image, fields, group_name, temperature=0.0)
    if not GEMINI_SELF_CONSISTENT:
        return {f: r1.get(f) for f in fields}, {f: "OK" for f in fields}

    r2 = call_gemini_group(pil_image, fields, group_name, temperature=0.35)
    final, flags = {}, {}
    for f in fields:
        v1 = str(r1.get(f) or "").strip()
        v2 = str(r2.get(f) or "").strip()
        if v1 == v2:
            final[f] = r1.get(f)
            flags[f] = "OK"
        else:
            final[f] = r1.get(f) if v1 else r2.get(f)
            flags[f] = "MISMATCH"
    return final, flags


In [ ]:
# Full catalog of fields that COULD be OpenCV-detected once calibrated.
# Only fields actually present in bubble_templates_blue.json are used at
# runtime; everything else automatically falls back to Gemini. Expand
# bubble_templates_blue.json (via the calibration tool below) to move more
# fields from Gemini to OpenCV without touching this notebook's code.
SCALE_4 = {
    "ARU":    ["ある", "ややある", "あまりない", "ない"],
    "KANJIRU":["感じる", "やや感じる", "あまり感じない", "感じない"],
    "OMOU":   ["思う", "やや思う", "あまり思わない", "思わない"],
    "TANOSHII": ["楽しい", "やや楽しい", "あまり楽しくない", "楽しくない"],
    "ITSUMO": ["いつもある", "だいたいある", "あまりない", "全くない"],
    "TORIKUMI": ["いつも取り組めていた", "だいたい取り組めていた", "あまり取り組めていなかった", "取り組めていなかった"],
}

OPENCV_SINGLE_CHOICE_CANDIDATES = {
    "性別": (["男", "女"], "header_measure"),
    "質1": (["好き", "やや好き", "やや嫌い", "嫌い"], "survey_1_7"),
    "質3": (["感じる", "やや感じる", "あまり感じない", "感じない"], "survey_1_7"),
    "質5": (["思う", "やや思う", "あまり思わない", "思わない"], "survey_1_7"),
    "質2-❶": (SCALE_4["ARU"], "survey_1_7"), "質2-❷": (SCALE_4["ARU"], "survey_1_7"),
    "質2-❸": (SCALE_4["ARU"], "survey_1_7"), "質2-❹": (SCALE_4["ARU"], "survey_1_7"),
    "質2-❺": (SCALE_4["ARU"], "survey_1_7"),
    "質4-❶": (SCALE_4["KANJIRU"], "survey_1_7"), "質4-❷": (SCALE_4["KANJIRU"], "survey_1_7"),
    "質4-❸": (SCALE_4["KANJIRU"], "survey_1_7"), "質4-❹": (SCALE_4["KANJIRU"], "survey_1_7"),
    "質4-❺": (SCALE_4["KANJIRU"], "survey_1_7"),
    "質6-❶": (SCALE_4["ARU"], "survey_1_7"), "質6-❷": (SCALE_4["ARU"], "survey_1_7"),
    "質6-❸": (SCALE_4["ARU"], "survey_1_7"),
    "質12": (SCALE_4["TANOSHII"], "health_class"),
    "質13": (SCALE_4["ITSUMO"], "health_class"), "質14": (SCALE_4["ITSUMO"], "health_class"),
    "質15": (SCALE_4["ITSUMO"], "health_class"), "質17": (SCALE_4["ITSUMO"], "health_class"),
    "質16": (SCALE_4["TORIKUMI"], "health_class"),
    "質18": (SCALE_4["OMOU"], "health_class"), "質19": (SCALE_4["OMOU"], "health_class"),
}

OPENCV_MULTI_CANDIDATES = {
    "質7-①": "survey_1_7", "質7-②": "survey_1_7", "質7-③": "survey_1_7",
    "質7-④": "survey_1_7", "質7-⑤": "survey_1_7",
    "質8-2-①": "club_time", "質8-2-②": "club_time", "質8-2-③": "club_time",
    "質8-2-④": "club_time", "質8-2-⑤": "club_time",
    "質12-2-①": "health_class", "質12-2-②": "health_class", "質12-2-③": "health_class",
    "質12-2-④": "health_class", "質12-2-⑤": "health_class", "質12-2-⑥": "health_class",
    "質12-2-⑦": "health_class", "質12-2-⑧": "health_class", "質12-2-⑨": "health_class",
    "質12-2-⑩": "health_class",
}

try:
    with open(BUBBLE_TEMPLATE_PATH, encoding="utf-8") as f:
        BUBBLE_TEMPLATES = json.load(f)
except FileNotFoundError:
    BUBBLE_TEMPLATES = {}

# Only calibrated fields are actually routed through OpenCV.
OPENCV_SINGLE_CHOICE = {f: v for f, v in OPENCV_SINGLE_CHOICE_CANDIDATES.items() if f in BUBBLE_TEMPLATES}
OPENCV_MULTI = {f: v for f, v in OPENCV_MULTI_CANDIDATES.items() if f in BUBBLE_TEMPLATES}
GEMINI_ONLY_FIELDS = [f for f in COLUMN_NAMES if f not in OPENCV_SINGLE_CHOICE and f not in OPENCV_MULTI]

print(f"OpenCV-calibrated: {len(OPENCV_SINGLE_CHOICE) + len(OPENCV_MULTI)} field(s)")
print(f"Gemini-only: {len(GEMINI_ONLY_FIELDS)} field(s)")


In [ ]:
def pil_to_gray(pil_img):
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2GRAY)

def frac_to_pixel(pil_roi, cx_f, cy_f, r_f):
    w, h = pil_roi.size
    cx, cy = int(cx_f * w), int(cy_f * h)
    r = max(4, int(r_f * min(w, h)))
    return cx, cy, r

def _circle_mask(h, w, cx, cy, r):
    Y, X = np.ogrid[:h, :w]
    return (X - cx) ** 2 + (Y - cy) ** 2 <= r ** 2

def fill_ratio(gray, cx, cy, r, dark_thresh):
    """Fraction of dark pixels inside the circle."""
    h, w = gray.shape
    mask = _circle_mask(h, w, cx, cy, r)
    if mask.sum() == 0:
        return 0.0
    region = gray[mask]
    return float((region < dark_thresh).sum()) / region.size

def bubble_confidence(ratios, chosen_idx, min_fill=0.12):
    best = ratios[chosen_idx]
    if best < min_fill:
        return 0.0
    others = [r for i, r in enumerate(ratios) if i != chosen_idx]
    second = max(others) if others else 0.0
    margin = best - second
    fill_score = min(1.0, (best - min_fill) / 0.40)
    margin_score = min(1.0, margin / 0.22)
    return round(0.55 * fill_score + 0.45 * margin_score, 3)

def detect_single_choice(gray, bubble_coords, labels):
    med = np.median(gray)
    dark_thresh = max(80, min(160, int(med - DARK_THRESH_OFFSET)))
    ratios = [fill_ratio(gray, cx, cy, r, dark_thresh) for cx, cy, r in bubble_coords]
    if not ratios or max(ratios) < 0.08:
        return None, 0.0, ratios
    idx = int(np.argmax(ratios))
    conf = bubble_confidence(ratios, idx)
    label = labels[idx] if idx < len(labels) else None
    return label, conf, ratios

def detect_multi_checkbox(gray, bubble_coord, dark_thresh=None):
    """Returns (0/1, confidence) for a single checkbox mark."""
    if dark_thresh is None:
        med = np.median(gray)
        dark_thresh = max(80, min(160, int(med - DARK_THRESH_OFFSET)))
    cx, cy, r = bubble_coord
    fr = fill_ratio(gray, cx, cy, r, dark_thresh)
    if fr >= 0.30:
        return "1", min(1.0, fr / 0.55)
    if fr <= 0.10:
        return "0", min(1.0, (0.15 - fr) / 0.15)
    return ("1" if fr > 0.18 else "0"), 0.35


In [ ]:
def process_page_hybrid(full_img):
    rois = get_all_rois(full_img)
    record, flags, sources = {}, {}, {}
    fallback_fields_by_group = {}
    opencv_ok_fields = []

    for field, (labels, group_name) in OPENCV_SINGLE_CHOICE.items():
        coords_frac = BUBBLE_TEMPLATES.get(field)
        if not coords_frac or len(coords_frac) != len(labels):
            fallback_fields_by_group.setdefault(group_name, []).append(field)
            continue
        gray = pil_to_gray(rois[group_name])
        pixel_coords = [frac_to_pixel(rois[group_name], *c) for c in coords_frac]
        label, conf, _ = detect_single_choice(gray, pixel_coords, labels)
        if conf >= OPENCV_CONF_THRESHOLD and label is not None:
            record[field], flags[field], sources[field] = label, "OK", f"opencv({conf:.2f})"
            opencv_ok_fields.append(field)
        else:
            fallback_fields_by_group.setdefault(group_name, []).append(field)

    for field, group_name in OPENCV_MULTI.items():
        coords_frac = BUBBLE_TEMPLATES.get(field)
        if not coords_frac:
            fallback_fields_by_group.setdefault(group_name, []).append(field)
            continue
        gray = pil_to_gray(rois[group_name])
        c = coords_frac[0] if isinstance(coords_frac[0], (list, tuple)) else coords_frac
        pixel = frac_to_pixel(rois[group_name], *c)
        val, conf = detect_multi_checkbox(gray, pixel)
        if conf >= OPENCV_CONF_THRESHOLD:
            record[field], flags[field], sources[field] = val, "OK", f"opencv({conf:.2f})"
            opencv_ok_fields.append(field)
        else:
            fallback_fields_by_group.setdefault(group_name, []).append(field)

    gemini_plan = {}
    for f in GEMINI_ONLY_FIELDS:
        for gname, gfields in ALL_GROUPS:
            if f in gfields:
                gemini_plan.setdefault(gname, []).append(f)
                break
    for gname, flist in fallback_fields_by_group.items():
        gemini_plan.setdefault(gname, [])
        for f in flist:
            if f not in gemini_plan[gname]:
                gemini_plan[gname].append(f)

    for group_name, fields in gemini_plan.items():
        if not fields:
            continue
        final, fl = call_group_self_consistent(rois[group_name], fields, group_name)
        for f in fields:
            record[f] = final.get(f)
            flags[f] = fl.get(f, "OK")
            sources[f] = sources.get(f, "gemini")

    return record, flags, sources


## Run pipeline on all PDFs in `PDF_INPUT_DIR`

In [ ]:
pdf_files = sorted(glob.glob(os.path.join(PDF_INPUT_DIR, "*.pdf")))
print(f"Found {len(pdf_files)} PDF file(s) in {PDF_INPUT_DIR}")

all_final, all_flags, all_sources = [], [], []

for pdf_path in pdf_files:
    doc = fitz.open(pdf_path)
    n_pages = len(doc)
    doc.close()
    print(f"\n{os.path.basename(pdf_path)} - {n_pages} page(s)")

    for page_num in range(n_pages):
        t0 = time.time()
        full_img = render_page(pdf_path, page_num)
        record, flag_record, source_record = process_page_hybrid(full_img)

        n_mismatch = sum(1 for v in flag_record.values() if v == "MISMATCH")
        n_opencv = sum(1 for v in source_record.values() if str(v).startswith("opencv"))
        record["_source_pdf"] = os.path.basename(pdf_path)
        record["_page"] = page_num
        record["_n_mismatch"] = n_mismatch

        all_final.append(record)
        all_flags.append(flag_record)
        all_sources.append(source_record)

        print(f"   page {page_num}: No.={record.get('No.')} "
              f"({time.time()-t0:.1f}s, mismatch={n_mismatch}, opencv={n_opencv})")

print(f"\nDone. {len(all_final)} record(s) processed.")


## Export results to Excel

In [ ]:
timestamp = time.strftime("%Y%m%d_%H%M%S")
out_path = os.path.join(OUT_DIR, f"体力測定調査結果_hybrid_{MODEL_NAME}_{timestamp}.xlsx")

df_final = pd.DataFrame(all_final)
df_flags = pd.DataFrame(all_flags)
df_sources = pd.DataFrame(all_sources)

for col in COLUMN_NAMES:
    if col not in df_final.columns:
        df_final[col] = None
    if col not in df_flags.columns:
        df_flags[col] = None

df_final = df_final[COLUMN_NAMES + [c for c in ("_source_pdf", "_page", "_n_mismatch") if c in df_final.columns]]
df_flags = df_flags.reindex(columns=COLUMN_NAMES)

wb = openpyxl.Workbook()
wb.remove(wb.active)

def append_df(wb, df, name):
    ws = wb.create_sheet(title=name)
    ws.append(list(df.columns))
    for row in df.fillna("").astype(str).values.tolist():
        ws.append(row)
    return ws

ws_final = append_df(wb, df_final, "納品データ")
ws_flags = append_df(wb, df_flags, "自己一致性チェック")
if not df_sources.empty:
    append_df(wb, df_sources.reindex(columns=COLUMN_NAMES), "ソース_opencv_gemini")

red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
red_font = Font(color="9C0006", bold=True)
for r in range(2, len(df_final) + 2):
    for c, col in enumerate(COLUMN_NAMES, start=1):
        if r - 2 < len(df_flags) and df_flags.iloc[r - 2].get(col) == "MISMATCH":
            cell = ws_final.cell(row=r, column=c)
            cell.fill = red_fill
            cell.font = red_font

wb.save(out_path)
print(f"Saved: {out_path}")


## Optional: calibrate more fields for OpenCV

Only fields listed in `bubble_templates_blue.json` are handled by OpenCV; the
rest already work fine through Gemini. Use this tool only if you want to move
more fields to OpenCV for speed/cost reasons - it is not required.

1. Set `field_name` below to the field you want to calibrate (e.g. `"質2-❶"`).
2. Run this cell, then click each bubble **in the exact printed order** shown in the title.
3. Run the next cell to save the click into `bubble_templates_blue.json`.
4. Repeat for other fields, or check `checklist` cell to see what's left.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

pdf_files = sorted(glob.glob(os.path.join(PDF_INPUT_DIR, "*.pdf")))
if not pdf_files:
    raise FileNotFoundError(f"No PDF found in {PDF_INPUT_DIR}. Add a sample PDF first.")

sample_img = render_page(pdf_files[0], 0)
rois = get_all_rois(sample_img)

# Change this to the field you want to calibrate, then re-run this cell.
field_name = "質2-❶"

if field_name in OPENCV_SINGLE_CHOICE_CANDIDATES:
    labels, roi_name = OPENCV_SINGLE_CHOICE_CANDIDATES[field_name]
    n_expected = len(labels)
elif field_name in OPENCV_MULTI_CANDIDATES:
    roi_name = OPENCV_MULTI_CANDIDATES[field_name]
    labels = [field_name]
    n_expected = 1
else:
    raise ValueError(f"{field_name} is not in OPENCV_SINGLE_CHOICE_CANDIDATES or OPENCV_MULTI_CANDIDATES")

roi = rois[roi_name]
gray = pil_to_gray(roi)

fig, ax = plt.subplots(figsize=(6, 8), constrained_layout=True)
ax.imshow(gray, cmap="gray")
ax.set_title(f"Click in order: {labels}  |  ROI: {roi_name}")
ax.set_axis_off()
try:
    fig.canvas.layout.width = '800px'
    fig.canvas.layout.height = '800px'
except Exception:
    pass

clicks = []

def onclick(event):
    if event.xdata is None:
        return
    x, y = event.xdata, event.ydata
    w, h = roi.size
    cx_f, cy_f = x / w, y / h
    r_f = 0.012  # adjust if the bubble radius looks off
    clicks.append((round(cx_f, 4), round(cy_f, 4), r_f))
    ax.add_patch(Circle((x, y), r_f * min(w, h), fill=False, edgecolor="lime", linewidth=2))
    fig.canvas.draw()
    idx = len(clicks) - 1
    lbl = labels[idx] if idx < len(labels) else "?"
    print(f"  [{idx}] {lbl}: ({cx_f:.4f}, {cy_f:.4f}, {r_f})  # {len(clicks)}/{n_expected}")

cid = fig.canvas.mpl_connect("button_press_event", onclick)
print(f"Field: {field_name} - click {n_expected} point(s) in order: {labels}")


In [ ]:
if "clicks" not in globals():
    print("No clicks recorded. Run the calibration cell above first.")
elif len(clicks) != n_expected:
    print(f"Clicked {len(clicks)} point(s) but '{field_name}' needs exactly {n_expected} "
          f"({labels}). Not saved - re-run the calibration cell and click the right count.")
else:
    BUBBLE_TEMPLATES[field_name] = clicks
    with open(BUBBLE_TEMPLATE_PATH, "w", encoding="utf-8") as f:
        json.dump(BUBBLE_TEMPLATES, f, ensure_ascii=False, indent=2)
    print(f"Saved BUBBLE_TEMPLATES['{field_name}'] ({len(clicks)} point(s)) -> {BUBBLE_TEMPLATE_PATH}")
    print("Change field_name in the calibration cell to the next field, then re-run both cells.")
    OPENCV_SINGLE_CHOICE = {f: v for f, v in OPENCV_SINGLE_CHOICE_CANDIDATES.items() if f in BUBBLE_TEMPLATES}
    OPENCV_MULTI = {f: v for f, v in OPENCV_MULTI_CANDIDATES.items() if f in BUBBLE_TEMPLATES}
    GEMINI_ONLY_FIELDS = [f for f in COLUMN_NAMES if f not in OPENCV_SINGLE_CHOICE and f not in OPENCV_MULTI]


### Calibration checklist

In [ ]:
_need = list(OPENCV_SINGLE_CHOICE_CANDIDATES.keys()) + list(OPENCV_MULTI_CANDIDATES.keys())
_done = [f for f in _need if f in BUBBLE_TEMPLATES]
_todo = [f for f in _need if f not in BUBBLE_TEMPLATES]
print(f"Calibrated: {len(_done)}/{len(_need)}")
if _todo:
    print("Not yet calibrated (using Gemini):")
    for f in _todo:
        print(" -", f)
else:
    print("All candidate fields are calibrated.")
